# FloorPlanCAD Object Detection — Colab Training
**Trước khi chạy**: `Runtime` → `Change runtime type` → **GPU (T4 hoặc A100)**

### Cấu trúc storage
```
/content/              ← Colab SSD (nhanh, tạm thời — mất khi session hết)
  object_detection/    ← code (git clone mỗi session)
  FloorPlanCAD_orig/   ← raw data (unzip từ Drive vào đây)
  FloorPlanCAD_data/   ← processed dataset (build vào đây)

Drive/MyDrive/Research/multi_model/object_detection/
  data/zips/           ← zip gốc (download 1 lần, giữ mãi)
  data/dataset.tar.gz  ← processed dataset đã tar (build 1 lần, giữ mãi)
  checkpoints/         ← model checkpoints
```

## Cell 1 — Setup: GPU + Drive + Clone

In [ ]:
# Kiểm tra GPU
import torch
print(f'GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — check runtime type!"}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/Research/multi_model/object_detection'
os.makedirs(f'{WORK_DIR}/data/zips', exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
print(f'Work dir: {WORK_DIR}')

# Clone code
%cd /content
!git clone https://github.com/DaniYLab/object_detection.git 2>/dev/null || (cd object_detection && git pull)
%cd /content/object_detection
!ls

## Cell 2 — Install dependencies

In [ ]:
# Colab đã có torch CUDA — chỉ cài thêm thư viện khác
!pip install -q gdown Pillow tqdm

import torch
print(f'torch : {torch.__version__}')
print(f'cuda  : {torch.version.cuda}')

## Cell 3A — LẦN ĐẦU: Download zip gốc về Drive
> ⏭️ Skip nếu đã có `Drive/.../data/zips/` rồi

In [ ]:
# Download 3 zip file vào Drive (file lớn → OK ghi thẳng vào Drive)
# Tạo symlink data/ → Drive để script download đúng chỗ
!rm -f /content/object_detection/data
!ln -sf "{WORK_DIR}/data" /content/object_detection/data

!python scripts/data/download_gdrive.py
# Kết quả: Drive/.../data/FloorPlanCAD_original/zips/*.zip

## Cell 3B — LẦN ĐẦU: Build dataset (toàn bộ trên SSD, nhanh)
> ⏭️ Skip nếu đã có `Drive/.../data/dataset.tar.gz` rồi

In [ ]:
import os

# Bước 1: Unzip raw data từ Drive vào SSD /content/ (đọc nhanh từ Drive, write nhanh vào SSD)
!mkdir -p /content/FloorPlanCAD_orig
print('Extracting train_set_1...')
!unzip -q "{WORK_DIR}/data/FloorPlanCAD_original/zips/train_set_1.zip" -d /content/FloorPlanCAD_orig/
print('Extracting train_set_2...')
!unzip -q "{WORK_DIR}/data/FloorPlanCAD_original/zips/train_set_2.zip" -d /content/FloorPlanCAD_orig/
print('Extracting test_set...')
!unzip -q "{WORK_DIR}/data/FloorPlanCAD_original/zips/test_set.zip"    -d /content/FloorPlanCAD_orig/
print('Done extracting!')
!ls /content/FloorPlanCAD_orig/

In [ ]:
# Bước 2: Build crops + metadata — hoàn toàn trên SSD
os.environ['ORIGINAL_ROOT'] = '/content/FloorPlanCAD_orig'
os.environ['OUTPUT_ROOT']   = '/content/FloorPlanCAD_data'
os.environ['DATASET_ROOT']  = '/content/FloorPlanCAD_data'

print('Building dataset...')
!python scripts/data/build_dataset.py

print('\nGenerating bbox metadata...')
!python scripts/data/generate_metadata.py

In [ ]:
# Bước 3: Tar và lưu sang Drive (1 file duy nhất, chỉ làm 1 lần)
print('Creating tar archive...')
!tar -czf /content/dataset.tar.gz -C /content FloorPlanCAD_data

print('Copying to Drive...')
!mv /content/dataset.tar.gz "{WORK_DIR}/data/dataset.tar.gz"

SIZE = os.path.getsize(f'{WORK_DIR}/data/dataset.tar.gz') / 1e9
print(f'Saved! Size: {SIZE:.2f} GB')

## Cell 4 — Mỗi session: Load dataset từ Drive
> ✅ Chạy cell này mỗi lần mở session mới (thay cho Cell 3A + 3B)

In [ ]:
import os

# Extract dataset từ Drive về SSD (~vài phút)
if not os.path.exists('/content/FloorPlanCAD_data'):
    print('Extracting dataset from Drive to SSD...')
    !tar -xzf "{WORK_DIR}/data/dataset.tar.gz" -C /content/
    print('Done!')
else:
    print('Dataset already on SSD.')

# Trỏ data/ trong repo → /content/FloorPlanCAD_data
!rm -f /content/object_detection/data
!mkdir -p /content/object_detection/data
!ln -sf /content/FloorPlanCAD_data /content/object_detection/data/FloorPlanCAD_dataset

!ls /content/object_detection/data/

## Cell 5 — Verify dataset + model

In [ ]:
import sys
sys.path.insert(0, '/content/object_detection')

from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

train_ds = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='train', image_size=512)
val_ds   = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='test',  image_size=512)

print(f'Train : {len(train_ds):,} samples')
print(f'Val   : {len(val_ds):,} samples')
print(f'Classes: {NUM_CLASSES}')

s = train_ds[0]
print(f'Image  : {s["image"].shape}')
print(f'Heatmap: {s["heatmap"].shape}')
print(f'Classes in sample: {[CLASS_NAMES[i] for i in s["class_ids"]]}')

# Verify model
import torch
from src.models.detector import FloorPlanDetector
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FloorPlanDetector(image_size=512, model_dim=512, num_classes=NUM_CLASSES, num_blocks=4).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\nModel params: {n_params:.1f}M on {device}')

img  = torch.randn(2, 3, 512, 512).to(device)
tids = torch.randint(0, 32000, (2, 16)).to(device)
cids = torch.tensor([3, 10]).to(device)
out  = model(img, tids, cids)
print(f'Output: {out.shape}  ✓')

## Cell 6 — Training

| GPU | VRAM | batch_size |
|-----|------|------------|
| T4  | 16GB | 8          |
| L4  | 24GB | 12         |
| A100| 40GB | 24         |

In [ ]:
import os

CKPT_DIR = f'{WORK_DIR}/checkpoints'   # lưu checkpoint trên Drive
os.makedirs(CKPT_DIR, exist_ok=True)

!python train.py \
    --data_root    ./data/FloorPlanCAD_dataset \
    --ckpt_dir     {CKPT_DIR} \
    --image_size   512 \
    --batch_size   8 \
    --num_workers  2 \
    --model_dim    512 \
    --num_blocks   4 \
    --epochs       50 \
    --lr           1e-4 \
    --log_interval 50

## Cell 7 — Load checkpoint & Visualize

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from src.models.detector import FloorPlanDetector
from src.data.dataset import FloorPlanDataset, CLASS_NAMES, NUM_CLASSES

CKPT_PATH = f'{WORK_DIR}/checkpoints/best.pt'
ckpt  = torch.load(CKPT_PATH, map_location='cpu')
model = FloorPlanDetector(image_size=512, model_dim=512, num_classes=NUM_CLASSES, num_blocks=4)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Epoch {ckpt["epoch"]} | Val IoU: {ckpt["val_iou"]:.4f}')

ds = FloorPlanDataset('./data/FloorPlanCAD_dataset', split='test', image_size=512)
sample = ds[0]

img_t  = sample['image'].unsqueeze(0)
tids   = torch.randint(0, 32000, (1, 16))
cids   = torch.tensor([sample['class_ids'][0] if sample['class_ids'] else 0])

with torch.no_grad():
    pred = model(img_t, tids, cids)[0]  # [35, h, w]

img_np = ((sample['image'].permute(1,2,0).numpy() * 0.5 + 0.5) * 255).astype('uint8')
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0,0].imshow(img_np); axes[0,0].set_title('Original'); axes[0,0].axis('off')

top_cls = pred.flatten(1).max(1).values.topk(7).indices
for i, cid in enumerate(top_cls):
    ax = axes[(i+1)//4, (i+1)%4]
    ax.imshow(pred[cid].numpy(), cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'{CLASS_NAMES[cid]} ({pred[cid].max():.2f})')
    ax.axis('off')

plt.tight_layout()
plt.savefig('prediction.png', dpi=150)
plt.show()